In [1]:
import pandas as pd
import joblib
import os

# Load raw datasets
price_train = pd.read_csv("data/price_train_data.csv", parse_dates=["Date"])
weather_train = pd.read_csv("data/weather_train_data.csv", parse_dates=["Date"])
price_eval = pd.read_csv("data/price_eval_data.csv", parse_dates=["Date"])
weather_eval = pd.read_csv("data/weather_eval_data.csv", parse_dates=["Date"])

# Rename columns for consistency
def clean_columns(price_df, weather_df):
    price_df.rename(columns={"Price per Unit (Silver Drachma/kg)": "price"}, inplace=True)
    weather_df.rename(columns={
        "Temperature (K)": "temperature",
        "Rainfall (mm)": "rainfall",
        "Humidity (%)": "humidity",
        "Crop Yield Impact Score": "yield_impact_score"
    }, inplace=True)

clean_columns(price_train, weather_train)
clean_columns(price_eval, weather_eval)

# Merge price and weather datasets
train = pd.merge(price_train, weather_train, on=["Date", "Region"], how="left")
eval = pd.merge(price_eval, weather_eval, on=["Date", "Region"], how="left")

# Resample to weekly average
def resample(df):
    df.set_index("Date", inplace=True)
    num_cols = df.select_dtypes(include="number").columns
    return df.groupby(["Region", "Commodity"])[num_cols].resample("W").mean().reset_index()

train = resample(train)
eval = resample(eval)

# Add lag features
def add_lags(df):
    for lag in [1, 2, 3, 4]:
        df[f"lag_{lag}"] = df.groupby(["Region", "Commodity"])["price"].shift(lag)
    return df

train = add_lags(train)
eval = add_lags(eval)

# Add rolling features
def add_rolling(df):
    df["rolling_mean_3"] = df.groupby(["Region", "Commodity"])["price"].transform(lambda x: x.rolling(3).mean())
    df["rolling_std_3"] = df.groupby(["Region", "Commodity"])["price"].transform(lambda x: x.rolling(3).std())
    return df

train = add_rolling(train)
eval = add_rolling(eval)

# Add price difference feature
train["lag_diff_1_2"] = train["lag_1"] - train["lag_2"]
eval["lag_diff_1_2"] = eval["lag_1"] - eval["lag_2"]

# Add date-based temporal features
def add_temporal_features(df):
    df["weekofyear"] = pd.to_datetime(df["Date"]).dt.isocalendar().week
    df["month"] = pd.to_datetime(df["Date"]).dt.month
    df["season"] = df["month"] % 12 // 3 + 1  # 1: Winter, 2: Spring, 3: Summer, 4: Fall
    return df

train = add_temporal_features(train)
eval = add_temporal_features(eval)

# Drop NaNs from lag/rolling features
train.dropna(inplace=True)
eval.dropna(inplace=True)

# Final features to be used in model
FEATURES = [
    "lag_1", "lag_2", "lag_3", "lag_4",
    "rolling_mean_3", "rolling_std_3", "lag_diff_1_2",
    "temperature", "rainfall", "humidity", "yield_impact_score",
    "weekofyear", "month", "season"
]
TARGET = "price"

X_train, y_train = train[FEATURES], train[TARGET]
X_eval, y_eval = eval[FEATURES], eval[TARGET]

# Save preprocessed files
os.makedirs("data", exist_ok=True)
joblib.dump(X_train, "data/X_train.pkl")
joblib.dump(y_train, "data/y_train.pkl")
joblib.dump(X_eval, "data/X_eval.pkl")
joblib.dump(y_eval, "data/y_eval.pkl")

print("✅ Enhanced features prepared and saved.")


✅ Enhanced features prepared and saved.
